In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

# Create a Partitioned Table

In [0]:
%sql
DROP TABLE IF EXISTS oc_catalog.oc_schema.snapshot_table;

CREATE TABLE IF NOT EXISTS oc_catalog.oc_schema.snapshot_table
(
    snapshot_date DATE,
    first_name STRING,
    last_name STRING,
    country STRING
)
PARTITIONED BY (snapshot_date)

# Create Dataset

In [0]:
# List of Data
snapshot_data = [
    (datetime.strptime("2025-09-30", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Canada"),
    (datetime.strptime("2025-09-30", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Mexico"),
    (datetime.strptime("2025-10-31", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Argentina"),
    (datetime.strptime("2025-10-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Cuba"),
    (datetime.strptime("2025-11-30", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Georgia"),
    (datetime.strptime("2025-12-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Germany")
]

# Define the Schema Explicitly
snapshot_schema = StructType([
    StructField("snapshot_date", DateType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df_snapshot = spark.createDataFrame(snapshot_data, snapshot_schema)

# Show the DataFrame and its Schema
df_snapshot.display()
df_snapshot.printSchema()

# Insert Data from Dataset into Partitioned Table Using PySpark for the First Time

In [0]:
df_snapshot\
        .write\
        .partitionBy("snapshot_date")\
        .mode("append")\
        .saveAsTable("oc_catalog.oc_schema.snapshot_table")

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.snapshot_table;

# Create Dataset to Append Data into a Partitioned Table

In [0]:
# List of Data
snapshot_data = [
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "India"),
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "India")
]

# Define the Schema Explicitly
snapshot_schema = StructType([
    StructField("snapshot_date", DateType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df_snapshot = spark.createDataFrame(snapshot_data, snapshot_schema)

# Show the DataFrame and its Schema
df_snapshot.display()
df_snapshot.printSchema()

# Fetch the Last Date of the Last Month

In [0]:
last_date_of_last_month = spark.sql("SELECT last_day(add_months(current_date(), -1))").collect()[0][0]
print(last_date_of_last_month)
print(type(last_date_of_last_month)) # It is a Date Type Variable

# Append Data from Source Dataset into the Existing Partitioned Table Using DataFrame Only If the Value of the Variable 'last_date_of_last_month' is Greater Than the Maximum Value of the Column "Snapshot_date"

In [0]:
max_snapshot_date = spark.sql("SELECT MAX(snapshot_date) FROM oc_catalog.oc_schema.snapshot_table").collect()[0][0]
print(max_snapshot_date)
print(type(max_snapshot_date))

if max_snapshot_date == last_date_of_last_month:
    df_snapshot.write\
            .format("delta")\
            .option("replaceWhere", f"snapshot_date = '{max_snapshot_date}'")\
            .mode("overwrite")\
            .saveAsTable("oc_catalog.oc_schema.snapshot_table")
    print(f"Overwriting Data for the Existing Partition '{max_snapshot_date}' into Snapshot Table 'oc_catalog.oc_schema.snapshot_table'")
elif max_snapshot_date < last_date_of_last_month:
    df_snapshot.write\
            .format("delta")\
            .mode("append")\
            .partitionBy("snapshot_date")\
            .saveAsTable("oc_catalog.oc_schema.snapshot_table")
    print(f"Appending Data for the New Partition '{last_date_of_last_month}' into Snapshot Table 'oc_catalog.oc_schema.snapshot_table'")

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.snapshot_table;